In [36]:
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda


## Chain with parrallel

In [37]:
# TASK 1- prompt 
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'Bạn là 1 chuyên gia'),
    ('human', '{message}')
])

In [38]:
# TASK 2 - LLM
llm = ChatOllama(model = 'llama3.2:3b', temperature=0)

In [39]:
# TASK 3 - OUTPUT PARSER
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()

In [40]:
# TASK 4 - [custom runnable]
from langchain_core.runnables import RunnableLambda
def dictionary_marker(text:str) -> dict:
    return {'text': text}
dictionary_marker_runnable = RunnableLambda(dictionary_marker)

## Parallel chain 1

In [41]:
# TASK 1 - PROMPT
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một biên tập viên chuyên nghiệp."),
    ("human", "Hãy tóm tắt lại nội dung sau trong 1 câu ngắn gọn:{text}")
])

 # TASK 2 - LLM
llm = ChatOllama(model = 'llama3.2:3b', temperature=0)

# TASK 3 - OUTPUT PARSER
str_parser = StrOutputParser()

chain_summary = prompt_summary | llm | str_parser

## Parallel chain 2

In [42]:
def chain_sentiment(text:dict):

    text = text['text']
    # TASK 1 - PROMPT
    prompt_sentiment = ChatPromptTemplate.from_messages([
        ("system", "Bạn là chuyên gia phân tích phản hồi khách hàng."),
        ("human", "Hãy đánh giá sắc thái của đoạn văn sau (Tích cực / Tiêu cực / Trung tính) kèm lý do ngắn: {text}")
    ])
    # TASK 2 - LLM
    llm = ChatOllama(model = 'llama3.2:3b', temperature=0)

    # TASK 3 - OUTPUT PARSER
    str_parser = StrOutputParser()

    chain_sentiment = prompt_sentiment | llm | str_parser

    result = chain_sentiment.invoke(text)

    return result

chain_sentiment_lambda = RunnableLambda(chain_sentiment)

## FINAL CHAIN

In [43]:
final_chain = prompt_template | llm | str_parser | dictionary_marker_runnable \
                    | RunnableParallel(
                        branch = {
                            'summary' : chain_summary,
                            'sentiment' : chain_sentiment_lambda
                        }
                    )

In [44]:
final_chain.invoke('Tôi vừa mua chiếc điện thoại này hôm qua. Thiết kế máy rất đẹp, màn hình sắc nét và pin cực kỳ trâu. Tuy nhiên, camera chụp đêm bị nhiễu hạt khá nhiều và giá bán thì hơi cao so với cấu hình')

{'branch': {'summary': 'Điện thoại của bạn có thiết kế và màn hình đẹp, nhưng camera chụp đêm có thể cần được cải thiện để đạt được chất lượng ảnh chụp đêm tốt hơn và giá bán cũng hơi cao so với cấu hình.',
  'sentiment': 'Đáp án: Trung tính\n\nLý do: Đoạn văn này có cả những điểm tích cực (thiết kế và màn hình đẹp, giá bán có thể được so sánh với các lựa chọn khác) và những điểm tiêu cực (camera chụp đêm có vấn đề về nhiễu hạt, giá bán hơi cao). Tác giả đã cố gắng đưa ra những nhận xét khách quan và đề xuất giải pháp, điều này cho thấy rằng đoạn văn này không thiên vị và trung thực.'}}

## CHAIN AS RUNNABLE

In [47]:
#TASK 1 -  beautify output
def beautify_output(final_chain : dict) -> dict:
    summary_response = final_chain['branch']['summary']
    sentiment_response = final_chain['branch']['sentiment']
    return {'summary': summary_response, 'sentiment': sentiment_response}

beautify_runnable = RunnableLambda(beautify_output)

# TASK 2 - output

beautify_output = final_chain | beautify_runnable

beautify_output.invoke('Tôi vừa mua chiếc điện thoại này hôm qua. Thiết kế máy rất đẹp, màn hình sắc nét và pin cực kỳ trâu. Tuy nhiên, camera chụp đêm bị nhiễu hạt khá nhiều và giá bán thì hơi cao so với cấu hình')

{'summary': 'Điện thoại của bạn có thiết kế và màn hình đẹp, nhưng camera chụp đêm có thể cần được cải thiện để đạt được chất lượng ảnh chụp đêm tốt hơn và giá bán cũng hơi cao so với cấu hình.',
 'sentiment': 'Đáp án: Trung tính\n\nLý do: Đoạn văn này có cả những điểm tích cực (thiết kế và màn hình đẹp, giá bán có thể được so sánh với các lựa chọn khác) và những điểm tiêu cực (camera chụp đêm có vấn đề về nhiễu hạt, giá bán hơi cao). Tác giả đã cố gắng đưa ra những nhận xét khách quan và đề xuất giải pháp, điều này cho thấy rằng đoạn văn này không thiên vị và trung thực.'}

In [ ]:
#TASK 1 -  beautify output
def beautify_output(final_chain : dict) -> dict:
    summary_response = final_chain['branch']['summary']
    sentiment_response = final_chain['branch']['sentiment']
    return {'summary': summary_response, 'sentiment': sentiment_response}

beautify_runnable = RunnableLambda(beautify_output)

# TASK 2 - output

beautify_output = final_chain | beautify_runnable

beautify_output.invoke('Tôi vừa mua chiếc điện thoại này hôm qua. Thiết kế máy rất đẹp, màn hình sắc nét và pin cực kỳ trâu. Tuy nhiên, camera chụp đêm bị nhiễu hạt khá nhiều và giá bán thì hơi cao so với cấu hình')

{'summary': 'Điện thoại của bạn có thiết kế và màn hình đẹp, nhưng camera chụp đêm có thể cần được cải thiện để đạt được chất lượng ảnh chụp đêm tốt hơn và giá bán cũng hơi cao so với cấu hình.',
 'sentiment': 'Đáp án: Trung tính\n\nLý do: Đoạn văn này có cả những điểm tích cực (thiết kế và màn hình đẹp, giá bán có thể được so sánh với các lựa chọn khác) và những điểm tiêu cực (camera chụp đêm có vấn đề về nhiễu hạt, giá bán hơi cao). Tác giả đã cố gắng đưa ra những nhận xét khách quan và đề xuất giải pháp, điều này cho thấy rằng đoạn văn này không thiên vị và trung thực.'}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda

# 1. Dữ liệu đầu vào (Input chứa đoạn code thô)
user_input = {
    "code": """
def get_even_numbers(numbers):
    result = []
    for i in range(len(numbers)):
        if numbers[i] % 2 == 0:
            result.append(numbers[i])
    return result
"""
}

# 2. Nhánh 1: Tìm lỗi / Đánh giá code
prompt_review = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một Senior Code Reviewer khó tính."),
    ("human", "Hãy nhận xét ngắn gọn về đoạn code Python sau xem nó có điểm gì chưa tối ưu hoặc cần cải thiện:\n\n{code}")
])

chain_review = prompt_review | ChatOllama(model='llama3.2:3b', temperature=0) | StrOutputParser()

# 3. Nhánh 2: Viết lại code tối ưu hơn (Dùng hàm def + RunnableLambda cho bạn luyện tập)
def refactor_code_func(input_data: dict):
    source_code = input_data['code']
    
    prompt_refactor = ChatPromptTemplate.from_messages([
        ("system", "Bạn là chuyên gia Python tối ưu hiệu năng."),
        ("human", "Hãy viết lại đoạn code sau theo chuẩn Pythonic (ngắn gọn, tối ưu hơn như dùng List Comprehension):\n\n{code}")
    ])
    
    llm = ChatOllama(model='llama3.2:3b', temperature=0)
    str_parser = StrOutputParser()
    
    refactor_chain = prompt_refactor | llm | str_parser
    return refactor_chain.invoke({"code": source_code})

# Biến hàm def thành RunnableLambda
chain_refactor_lambda = RunnableLambda(refactor_code_func)

# 4. Gom cả 2 vào Parallel Chain
parallel_code_chain = RunnableParallel(
    review=chain_review,
    refactor=chain_refactor_lambda
)

# 5. Chạy thử và in kết quả
result = parallel_code_chain.invoke(user_input)

print("--- ĐÁNH GIÁ CODE (REVIEW) ---")
print(result["review"])

print("\n--- CODE TỐI ƯU HƠN (REFACTOR) ---")
print(result["refactor"])